In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", google_api_key=api_key, temperature=0.7)

c:\FAHEEM\My_Programs\RAG_Beginners\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Google api key is set


In [4]:
from typing import TypedDict, List, Annotated
from langgraph.graph.message import add_messages, BaseMessage
from langchain_core.documents import Document
from langchain_core.messages import ToolMessage

class graph_schema(TypedDict):
    question: Annotated[List[BaseMessage], add_messages]
    document: List[Document]
    answer: str

In [3]:
tools = []

llm_with_tools = llm.bind_tools(tools)

In [ ]:
def llm_node(state: graph_schema) -> graph_schema:
    question = state["question"]
    result = llm_with_tools.invoke(question)
    return {
        "question": result
    }

def tool_node(state: graph_schema) -> graph_schema:
    query = state['question']

    tool_by_name = {tool.name: tool for tool in tools}

    tool_result = []

    for tool_calls in query[-1].tool_calls:
        tool = tool_by_name[tool_calls["name"]]
        observation = tool.invoke(tool_calls["args"])
        tool_result.append(ToolMessage(content=str(observation), tool_call_id=tool_calls["id"]))

    return {
        "document": tool_result
    }
